# DataType - JavaScript

All 9 JavaScript examples from [docs/datatype.md](https://platob.github.io/yggdryl/datatype/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const { DataType } = require('yggdryl')

const value = DataType.from('decimal(18, 4)')
assert.equal(value.id, 'decimal128')

assert.equal(value.toString(), 'decimal128(18,4)')
assert.ok(DataType.fromString(value.toString()).equals(value))
assert.ok(DataType.fromJSON(value.toJSON()).equals(value))

## Children

In [ ]:
const assert = require('node:assert/strict')
const { DataType, fields } = require('yggdryl')

const quote = DataType.fromFields([
  fields.utf8('symbol'),
  fields.list('levels', fields.float64('item'), { nullable: true }),
])

assert.equal(quote.length, 2)
assert.deepEqual(quote.keys(), ['symbol', 'levels'])
assert.equal(quote.at(-1).name, 'levels')
assert.equal(quote.get('levels').dataType.length, 1)
assert.equal(quote.contains('missing'), false)
assert.deepEqual([...quote].map((field) => field.name), ['symbol', 'levels'])

const lookup = fields.mapOf('lookup', 'utf8', 'int64', true).dataType
assert.equal(lookup.length, 1)
assert.equal(lookup.at(0).name, 'entries')

## Precision and resolution pick the width

In [ ]:
const assert = require('node:assert/strict')
const { DataType, fields } = require('yggdryl')

assert.equal(fields.decimal('amount', 38, 4).dataType.toString(), 'decimal128(38,4)')
assert.equal(fields.decimal('wide', 39, 4).dataType.toString(), 'decimal256(39,4)')
assert.equal(DataType.time('s').toString(), 'time32(s)')
assert.equal(DataType.time('nano seconds').toString(), 'time64(ns)')

assert.throws(() => fields.decimal('bad', 2, 3), /positive scale cannot exceed precision/)
assert.throws(() => DataType.time('year_month'), /temporal resolution/)

## Encodings that wrap a value

In [ ]:
const assert = require('node:assert/strict')
const { fields } = require('yggdryl')

const codes = fields.dictionary('codes', 'int16', 'utf8').dataType
const runs = fields
  .runEndEncoded('runs', fields.int16('run_ends', { nullable: false }), fields.utf8('values'))
  .dataType

assert.equal(codes.kind, 'dictionary')
assert.equal(runs.kind, 'run_end_encoded')
assert.equal(codes.nested, false)
assert.equal(runs.nested, false)
assert.equal(codes.toString(), 'dictionary(int16,utf8)')

assert.throws(() => fields.dictionary('bad', 'utf8', 'utf8'), /integer key datatype/)
assert.throws(
  () => fields.runEndEncoded('bad', fields.uint32('run_ends', { nullable: false }), fields.utf8('values')),
  /int16, int32, or int64/,
)

## Unions and the variant alias

In [ ]:
const assert = require('node:assert/strict')
const { DataType, fields } = require('yggdryl')

const variant = DataType.variant([
  fields.int64('number'),
  fields.utf8('text', { nullable: true }),
])

assert.equal(variant.id, 'union')
assert.ok(variant.toString().startsWith('union(dense,'))
assert.deepEqual(variant.keys(), ['number', 'text'])
assert.equal(DataType.from('variant(number:int64,text:string)').id, 'union')

assert.throws(
  () => DataType.variant([fields.int64('same'), fields.utf8('same')]),
  /duplicate field name/,
)

## Identity and family

In [ ]:
const assert = require('node:assert/strict')
const { DataType, fields } = require('yggdryl')

const stamp = DataType.from('timestamp(ns, Europe/Paris)')
assert.equal(stamp.id, 'timestamp')
assert.equal(stamp.kind, 'temporal')

assert.equal(DataType.from('timestamp(s)').id, stamp.id)
assert.equal(DataType.from('timestamp(s)').equals(stamp), false)

assert.equal(fields.decimal('amount', 38, 4).dataType.id, 'decimal128')
assert.equal(fields.decimal('amount', 38, 4).dataType.kind, 'decimal')

## Arrow projection

In [ ]:
const assert = require('node:assert/strict')
const { DataType } = require('yggdryl')

// Any Apache Arrow JS type is read through its own textual form.
const arrowLike = { toString: () => 'map<string,array<decimal(38,18)>>' }
const value = DataType.fromArrow(arrowLike)

assert.equal(value.id, 'map')
assert.ok(DataType.fromArrow(value).equals(value))
assert.throws(() => DataType.fromArrow({}), /own textual representation/)

## Default values

In [ ]:
const assert = require('node:assert/strict')
const { DataType, fields } = require('yggdryl')

const value = DataType.fromFields([
  fields.int32('id', { nullable: false }),
  fields.utf8('note', { nullable: true }),
])

// A nullable field defaults to null; a required one defaults to its zero.
assert.deepEqual(value.defaultJSValue(), [0, null])
assert.equal(new DataType('utf8').defaultJSValue(), '')
assert.equal(new DataType('int64').defaultJSHint().constructor, BigInt)
assert.equal(new DataType('int32').defaultArrowScalar(), 0)

## Compatibility rewriting

In [ ]:
const assert = require('node:assert/strict')
const { DataType, fields } = require('yggdryl')

const source = DataType.fromFields([
  fields.uint8('small'),
  fields.uint64('wide', { nullable: true }),
])

const spark = source.toSchemeCompat('spark')
assert.equal(spark.get('small').dataType.toString(), 'int16')
assert.equal(spark.get('wide').dataType.toString(), 'decimal128(20,0)')

assert.ok(source.toSchemeCompat('arrow').equals(source))
assert.ok(DataType.from('uint32').toSchemeCompat('polars').equals(DataType.from('uint32')))

assert.throws(() => DataType.from('timestamp(ns)').toSchemeCompat('spark'), /got ns/)
assert.throws(
  () => DataType.from('int32').toSchemeCompat('duckdb'),
  /arrow, spark, polars, pandas/,
)